# 01 — Decision 2: Secured Lending Underwriting under Vendor Uncertainty

**Decision**: Should a bank approve a secured CRE/mortgage loan, and at what Basel III SA risk weight?

**Mechanism**: Physical flood damage reduces collateral value → raises current LTV → determines risk-weight bucket (standard / conditional / reject) → drives capital requirement.

**Key contrast with Decision 1**: Two LTV cliff thresholds (LTV̄₁ and LTV̄₂) replace the single SICR cliff. The transmission chain is shorter — no PD/LGD model — but the discontinuity structure is qualitatively similar: vendor model choice can tip a loan across a risk-weight bucket boundary.

```
damage_ratio (d)
  → V_iv  = V_i0 × (1 − θ × d)          impaired collateral value
  → LTV_iv = Loan_i / V_iv               current loan-to-value
  → outcome ∈ {standard, conditional, reject}   LTV bucket
  → RWA_i  = Loan_i × RW(outcome)        risk-weighted asset
  → Capital = capital_ratio × RWA
```

**Input**: `data/processed/portfolio.csv` (Notebook 02), `data/raw/cfrf_garp_defended_flood.csv`  
**Output**: figures → `outputs/figures/decision2/`, arrays → `outputs/`

In [1]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
import yaml

# Add src to path
sys.path.insert(0, str(Path(".").resolve().parent.parent / "src"))

from uncertainty.distributions import fit_all_distributions, sample_vendor_uncertainty
from secured_lending.transmission import (
    impaired_collateral_value, compute_ltv, classify_outcome,
    compute_rwa, compute_capital, analytical_damage_thresholds,
)
from utils.plotting import set_style, COLOURS

set_style()

# Load config
with open("../../config/parameters.yaml") as f:
    cfg = yaml.safe_load(f)

cfg_d2     = cfg["decision2_secured_lending"]
seed       = cfg_d2["random_seed"]
rng        = np.random.default_rng(seed)

# Output directories
Path("../../outputs/figures/decision2").mkdir(parents=True, exist_ok=True)
Path("../../outputs").mkdir(parents=True, exist_ok=True)

print("Config loaded. Decision 2 parameters:")
for k, v in cfg_d2.items():
    print(f"  {k}: {v}")

Config loaded. Decision 2 parameters:
  ltv_thresholds: {'ltv_bar_1': 0.6, 'ltv_bar_2': 0.8}
  risk_weights: {'rw_standard': 0.6, 'rw_conditional': 0.75, 'rw_reject': 1.05}
  theta: {'baseline': 1.0, 'sensitivity_values': [0.5, 0.75, 1.0]}
  capital_ratio: 0.08
  n_simulations: 10000
  random_seed: 42


## 2 — Load Data and Portfolio

Flood data: `data/raw/cfrf_garp_defended_flood.csv` — 100 UK properties, 13 vendors, 1-in-200 year defended combined flood, 2030.

Portfolio: `data/processed/portfolio.csv` — constructed by Notebook 02 using the LTV structural model. Decision 2 reuses the structural portfolio directly:
- `Loan_i = ead_m` (= LTV₀ × PV₀)
- `V_i0   = pv_0_m`
- `LTV_i0 = ltv_0` (informational; not used directly in D2 transmission)

In [2]:
# --- Flood data ---
df = pd.read_csv("../../data/raw/cfrf_garp_defended_flood.csv")
df.rename(columns={"minimim_dr": "minimum_dr"}, inplace=True)
df = df.dropna(subset=["property_rank"]).reset_index(drop=True)
df["property_rank"] = df["property_rank"].astype(int)
print(f"Properties loaded: {len(df)}")

# --- Portfolio ---
portfolio_path = Path("../../data/processed/portfolio.csv")
if not portfolio_path.exists():
    raise FileNotFoundError(
        "Portfolio not found. Run notebooks/shared/02_portfolio_construction.ipynb first."
    )
portfolio = pd.read_csv(portfolio_path)
print(f"Portfolio loaded: {len(portfolio)} loans")
print(f"Portfolio columns: {list(portfolio.columns)}")

# Decision 2 portfolio variables (structural reuse from D1)
loan_arr = portfolio["ead_m"].values        # Loan_i = EAD = LTV_0 * PV_0 (£m)
v0_arr   = portfolio["pv_0_m"].values       # V_i0 = property value at origination (£m)
ltv0_arr = portfolio["ltv_0"].values        # LTV_i0 = LTV at origination (informational)

# Convenience: damage ratio point estimates per property
d_min_arr  = df["minimum_dr"].values
d_mean_arr = df["mean_dr"].values
d_max_arr  = df["maximum_dr"].values

print(f"\nPortfolio summary:")
print(f"  Total loan book:  \u00a3{loan_arr.sum():.1f}m")
print(f"  Mean LTV\u2080:        {ltv0_arr.mean():.3f}  "
      f"(range: {ltv0_arr.min():.3f}\u2013{ltv0_arr.max():.3f})")
print(f"  Mean V\u2080:          \u00a3{v0_arr.mean():.2f}m")
print(f"  Mean loan (EAD):  \u00a3{loan_arr.mean():.3f}m")

Properties loaded: 100


FileNotFoundError: Portfolio not found. Run notebooks/shared/02_portfolio_construction.ipynb first.

## 3 — Vendor Uncertainty Distributions

Fit triangular distributions to damage ratios using `fit_all_distributions()` — identical to Decision 1 for cross-decision comparability. The triangular distribution is the primary choice: mode derived from `c = 3μ − min − max`, with clamping when the implied mode falls outside the support.

Zero-damage properties (all vendors report d = 0) are handled as point masses and excluded from distribution fitting; they contribute zero to all downstream metrics.

In [ ]:
# Fit triangular distributions (primary, matching D1)
fit_results = fit_all_distributions(df)

n_zero    = fit_results["is_zero_damage"].sum()
n_nonzero = (~fit_results["is_zero_damage"]).sum()
n_fitted  = fit_results["tri_success"].sum()

print("Distribution fitting (triangular):")
print(f"  Zero-damage properties (point mass at 0):  {n_zero}")
print(f"  Non-zero damage properties:                {n_nonzero}")
print(f"  Successful triangular fits:                {n_fitted}")
print(f"  Failed / degenerate fits:                  {n_nonzero - n_fitted}")

# Flag any clamped modes
n_clamped = fit_results["tri_clamped"].sum()
print(f"  Clamped modes (mean outside [min, max]):   {n_clamped}")

## 4 — Monte Carlo Simulation

Draw `n_simulations` × `n_properties` damage ratios from the fitted triangular distributions (independent draws across properties), then pass through the secured lending transmission chain.

The output arrays — `ltv_sim`, `outcome_sim`, `rwa_sim`, `capital_sim` — have shape (n_simulations, n_properties) and represent the full joint distribution of outcomes under vendor uncertainty.

In [ ]:
# --- Parameters from config ---
n_sim      = cfg_d2["n_simulations"]
theta      = cfg_d2["theta"]["baseline"]
ltv_bar_1  = cfg_d2["ltv_thresholds"]["ltv_bar_1"]
ltv_bar_2  = cfg_d2["ltv_thresholds"]["ltv_bar_2"]
rw_std     = cfg_d2["risk_weights"]["rw_standard"]
rw_cond    = cfg_d2["risk_weights"]["rw_conditional"]
rw_rej     = cfg_d2["risk_weights"]["rw_reject"]
cap_ratio  = cfg_d2["capital_ratio"]

print(f"Simulation: n_sim={n_sim:,}, \u03b8={theta}, seed={seed}")
print(f"LTV thresholds: LTV\u0305\u2081={ltv_bar_1}, LTV\u0305\u2082={ltv_bar_2}")
print(f"Risk weights: standard={rw_std}, conditional={rw_cond}, reject={rw_rej}")
print(f"Capital ratio: {cap_ratio}")

# Draw damage ratio samples: shape (n_sim, n_props)
d_sim = sample_vendor_uncertainty(
    df, fit_results,
    distribution="triangular",
    n_samples=n_sim,
    correlation="independent",
    random_state=seed,
)
print(f"\nDamage samples shape: {d_sim.shape}")

# --- Transmission chain ---
# Shapes broadcast: v0_arr/loan_arr (n_props,) × d_sim (n_sim, n_props)
v_imp_sim   = impaired_collateral_value(v0_arr, d_sim, theta)               # (n_sim, n_props)
ltv_sim     = compute_ltv(loan_arr, v_imp_sim)                              # (n_sim, n_props)
outcome_sim = classify_outcome(ltv_sim, ltv_bar_1, ltv_bar_2)               # (n_sim, n_props)
rwa_sim     = compute_rwa(loan_arr, outcome_sim, rw_std, rw_cond, rw_rej)   # (n_sim, n_props)
capital_sim = compute_capital(rwa_sim, cap_ratio)                           # (n_sim, n_props)

# Save arrays
np.save("../../outputs/d2_d_sim.npy",       d_sim)
np.save("../../outputs/d2_ltv_sim.npy",     ltv_sim)
np.save("../../outputs/d2_outcome_sim.npy",  outcome_sim)
np.save("../../outputs/d2_rwa_sim.npy",      rwa_sim)
np.save("../../outputs/d2_capital_sim.npy",  capital_sim)
print("Arrays saved to outputs/")

In [ ]:
# --- Calibration diagnostic ---
# At mean damage (theta=1), what fraction of assets cross each LTV threshold?
# If fewer than 5% are near either threshold, flag for recalibration.

v_imp_diag = impaired_collateral_value(v0_arr, d_mean_arr, theta=1.0)
ltv_diag   = compute_ltv(loan_arr, v_imp_diag)

frac_above_bar1 = (ltv_diag > ltv_bar_1).mean()
frac_above_bar2 = (ltv_diag > ltv_bar_2).mean()

print("=== Calibration diagnostic (\u03b8=1, mean damage) ===")
print(f"  Fraction with LTV > LTV\u0305\u2081 ({ltv_bar_1}): {frac_above_bar1:.1%}")
print(f"  Fraction with LTV > LTV\u0305\u2082 ({ltv_bar_2}): {frac_above_bar2:.1%}")
print()

if frac_above_bar1 < 0.05 and frac_above_bar2 < 0.05:
    print("WARNING: Fewer than 5% of assets cross either LTV threshold at mean damage.")
    print("  The LTV distribution may need recalibration for Decision 2.")
    print("  Consider: (a) higher baseline LTVs, (b) larger damage ratios, or")
    print("  (c) lower LTV thresholds for this CRE context.")
else:
    print("Calibration OK: sufficient asset near-threshold exposure at mean damage.")

print(f"\nLTV at mean damage (theta=1): "
      f"mean={ltv_diag.mean():.3f}, "
      f"p25={np.percentile(ltv_diag, 25):.3f}, "
      f"p75={np.percentile(ltv_diag, 75):.3f}, "
      f"max={ltv_diag.max():.3f}")

## 5 — Tier 1: Raw Vendor Disagreement

Property-level summary statistics of vendor spread, identical to Decision 1 for cross-decision comparability. These metrics characterise the *physical risk input uncertainty* before any financial transmission.

- **Relative spread**: `(d_max − d_min) / d_mean` — normalised width of vendor range
- **Skewness proxy**: `(d_mean − d_min) / (d_max − d_min)` — position of mean within range; 0.5 = symmetric, >0.5 = right-skewed vendors
- **Pessimism ratio**: `d_max / d_mean` — how many times more severe is the most pessimistic vendor

In [ ]:
non_zero_mask = ~fit_results["is_zero_damage"].values

# Relative spread: (d_max - d_min) / d_mean
with np.errstate(invalid="ignore", divide="ignore"):
    rel_spread = np.where(d_mean_arr > 0,
                          (d_max_arr - d_min_arr) / d_mean_arr, np.nan)

# Skewness proxy: position of mean in [min, max]
with np.errstate(invalid="ignore", divide="ignore"):
    span_arr   = d_max_arr - d_min_arr
    skew_proxy = np.where(span_arr > 0,
                          (d_mean_arr - d_min_arr) / span_arr, np.nan)

# Pessimism ratio: d_max / d_mean
with np.errstate(invalid="ignore", divide="ignore"):
    pessimism = np.where(d_mean_arr > 0, d_max_arr / d_mean_arr, np.nan)

tier1 = pd.DataFrame({
    "property_rank":    df["property_rank"],
    "d_min":            d_min_arr,
    "d_mean":           d_mean_arr,
    "d_max":            d_max_arr,
    "rel_spread":       rel_spread,
    "skew_proxy":       skew_proxy,
    "pessimism_ratio":  pessimism,
})

print(f"Tier 1 summary (non-zero properties, n={non_zero_mask.sum()}):")
print(tier1.loc[non_zero_mask, ["rel_spread", "skew_proxy", "pessimism_ratio"]]
      .describe().round(3))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax = axes[0]
ax.hist(tier1.loc[non_zero_mask, "rel_spread"].dropna(), bins=20,
        color=COLOURS["beta"], edgecolor="white", alpha=0.85)
ax.set_xlabel("Relative spread  (d_max \u2212 d_min) / d_mean")
ax.set_ylabel("Count")
ax.set_title("Tier 1a: Vendor relative spread")

ax = axes[1]
ax.hist(tier1.loc[non_zero_mask, "skew_proxy"].dropna(), bins=20,
        color=COLOURS["triangular"], edgecolor="white", alpha=0.85)
ax.axvline(0.5, color="k", ls="--", lw=1.2, label="Symmetric (=0.5)")
ax.set_xlabel("Skewness proxy  (d_mean \u2212 d_min) / (d_max \u2212 d_min)")
ax.set_ylabel("Count")
ax.set_title("Tier 1b: Implied skewness")
ax.legend()

ax = axes[2]
ax.hist(tier1.loc[non_zero_mask, "pessimism_ratio"].dropna(), bins=20,
        color=COLOURS["uniform"], edgecolor="white", alpha=0.85)
ax.set_xlabel("Pessimism ratio  d_max / d_mean")
ax.set_ylabel("Count")
ax.set_title("Tier 1c: Pessimism ratio")

plt.suptitle("Tier 1: Raw vendor disagreement", y=1.02, fontsize=13, fontweight="bold")
plt.tight_layout()
fig.savefig("../../outputs/figures/decision2/tier1_raw_disagreement.pdf", dpi=300)
fig.savefig("../../outputs/figures/decision2/tier1_raw_disagreement.png", dpi=300)
plt.show()
print("Saved: tier1_raw_disagreement.{pdf,png}")

## 6 — Tier 2: Decision-Propagation Metrics

These metrics characterise how vendor uncertainty propagates through the lending decision. The key question is: *how often does the vendor draw materially change which risk-weight bucket a loan falls into?*

- **Threshold-crossing probability**: `P(LTV_sim > LTV̄_k)` per asset per threshold
- **Decision Reversal Rate (DRR)**: `1 − max_k P(outcome = k)` — probability that a randomly drawn vendor does NOT place the loan in its modal bucket
- **Extreme-vendor reversal rate**: fraction of loans where outcome at `d_min` differs from outcome at `d_max`
- **Ambiguity zone**: loans where the analytical threshold `d*` falls within the vendor support `[d_min, d_max]`
- **Capital bucket instability**: fraction of MC draws where outcome ≠ mean-damage outcome
- **Cliff Proximity Index (CPI)**: how close is the mean-damage LTV to the nearest threshold, relative to the LTV simulation standard deviation

In [ ]:
# --- Threshold-crossing probabilities ---
p_cross_bar1 = (ltv_sim > ltv_bar_1).mean(axis=0)   # P(LTV > LTV_bar_1)
p_cross_bar2 = (ltv_sim > ltv_bar_2).mean(axis=0)   # P(LTV > LTV_bar_2)

# --- Decision Reversal Rate ---
p_outcome = np.stack([
    (outcome_sim == 0).mean(axis=0),   # P(standard)
    (outcome_sim == 1).mean(axis=0),   # P(conditional)
    (outcome_sim == 2).mean(axis=0),   # P(reject)
], axis=1)  # (n_props, 3)

drr = 1.0 - p_outcome.max(axis=1)     # (n_props,)

# Loan-weighted mean DRR
weights = loan_arr / loan_arr.sum()
drr_portfolio_mean = (drr * weights).sum()

print(f"Portfolio loan-weighted mean DRR: {drr_portfolio_mean:.4f}  ({drr_portfolio_mean:.2%})")
print(f"Asset-level DRR:  mean={drr.mean():.4f}  max={drr.max():.4f}  "
      f"n(DRR>5%)={( drr > 0.05).sum()}")

# --- Extreme-vendor reversal rate ---
outcome_at_dmin  = classify_outcome(
    compute_ltv(loan_arr, impaired_collateral_value(v0_arr, d_min_arr, theta)),
    ltv_bar_1, ltv_bar_2
)
outcome_at_dmean = classify_outcome(
    compute_ltv(loan_arr, impaired_collateral_value(v0_arr, d_mean_arr, theta)),
    ltv_bar_1, ltv_bar_2
)
outcome_at_dmax  = classify_outcome(
    compute_ltv(loan_arr, impaired_collateral_value(v0_arr, d_max_arr, theta)),
    ltv_bar_1, ltv_bar_2
)

extreme_reversal_rate = (outcome_at_dmin != outcome_at_dmax).mean()
print(f"Extreme-vendor reversal rate: {extreme_reversal_rate:.2%}  "
      f"({int(extreme_reversal_rate * len(loan_arr))} of {len(loan_arr)} loans)")

# --- Ambiguity zone ---
d_star_1 = analytical_damage_thresholds(loan_arr, v0_arr, ltv_bar_1, theta)
d_star_2 = analytical_damage_thresholds(loan_arr, v0_arr, ltv_bar_2, theta)

in_ambig_1 = (~np.isnan(d_star_1)) & (d_star_1 >= d_min_arr) & (d_star_1 <= d_max_arr)
in_ambig_2 = (~np.isnan(d_star_2)) & (d_star_2 >= d_min_arr) & (d_star_2 <= d_max_arr)
in_ambig_either = in_ambig_1 | in_ambig_2

print(f"\nAmbiguity zone (d* \u2208 [d_min, d_max]):")
print(f"  Threshold LTV\u0305\u2081={ltv_bar_1}: {in_ambig_1.sum()} assets")
print(f"  Threshold LTV\u0305\u2082={ltv_bar_2}: {in_ambig_2.sum()} assets")
print(f"  Either threshold:          {in_ambig_either.sum()} assets")

In [ ]:
# --- Capital bucket instability ---
bucket_instability = (outcome_sim != outcome_at_dmean[np.newaxis, :]).mean(axis=0)  # (n_props,)
print(f"Capital bucket instability (fraction of draws \u2260 mean-damage outcome):")
print(f"  Mean: {bucket_instability.mean():.4f}  Max: {bucket_instability.max():.4f}  "
      f"n(>5%): {(bucket_instability > 0.05).sum()}")

# --- Cliff Proximity Index ---
# CPI_i = 1 - min_k(|LTV_i(d_mean) - LTV_bar_k|) / std(LTV_i_sim), clipped to [0, 1]
ltv_at_dmean = compute_ltv(loan_arr, impaired_collateral_value(v0_arr, d_mean_arr, theta))
ltv_sim_std  = ltv_sim.std(axis=0)   # (n_props,)

dist_to_bar1    = np.abs(ltv_at_dmean - ltv_bar_1)
dist_to_bar2    = np.abs(ltv_at_dmean - ltv_bar_2)
dist_to_nearest = np.minimum(dist_to_bar1, dist_to_bar2)

with np.errstate(invalid="ignore", divide="ignore"):
    cpi = np.clip(1.0 - dist_to_nearest / ltv_sim_std, 0.0, 1.0)

# Key diagnostic: CPI vs DRR scatter
fig, ax = plt.subplots(figsize=(9, 6))
sc = ax.scatter(
    cpi, drr,
    c=loan_arr,
    cmap="YlOrRd",
    s=55, alpha=0.75, edgecolors="grey", linewidths=0.4, zorder=3,
)
plt.colorbar(sc, ax=ax, label="Loan size (\u00a3m)")
ax.set_xlabel("Cliff Proximity Index  (0 = far from threshold, 1 = on threshold)")
ax.set_ylabel("Decision Reversal Rate")
ax.set_title("Tier 2: CPI vs DRR — vendor uncertainty exposure by loan")
ax.axhline(0.05, color=COLOURS["threshold"], ls="--", lw=1, alpha=0.7,
           label="DRR = 5%")
ax.legend(fontsize=9)
plt.tight_layout()
fig.savefig("../../outputs/figures/decision2/tier2_cpi_drr_scatter.pdf", dpi=300)
fig.savefig("../../outputs/figures/decision2/tier2_cpi_drr_scatter.png", dpi=300)
plt.show()
print("Saved: tier2_cpi_drr_scatter.{pdf,png}")

print(f"\nTier 2 summary:")
print(f"  Portfolio DRR (loan-weighted):   {drr_portfolio_mean:.2%}")
print(f"  Extreme-vendor reversal rate:    {extreme_reversal_rate:.2%}")
print(f"  Assets in ambiguity zone:        {in_ambig_either.sum()} / {len(loan_arr)}")
print(f"  Mean bucket instability:         {bucket_instability.mean():.2%}")

## 7 — Tier 3: Portfolio Aggregates

Portfolio-level metrics that aggregate across all 100 loans. All figures are expressed as **ratios or percentage spreads relative to the mean-vendor baseline** (point estimate at `d_mean` with `θ = baseline`), not as absolute £ amounts.

- **RWA spread**: `(RWA at d_max − RWA at d_min) / RWA at d_mean` as %
- **Capital range**: % uplift from min-vendor to max-vendor capital requirement
- **Lorenz curve / Gini**: concentration of DRR across the loan book, weighted by EAD
- **Approval rate shift**: stacked bar at `d_min` / `d_mean` / `d_max`

In [ ]:
# --- Point-estimate RWAs ---
rwa_at_dmin  = compute_rwa(loan_arr, outcome_at_dmin,  rw_std, rw_cond, rw_rej)
rwa_at_dmean = compute_rwa(loan_arr, outcome_at_dmean, rw_std, rw_cond, rw_rej)
rwa_at_dmax  = compute_rwa(loan_arr, outcome_at_dmax,  rw_std, rw_cond, rw_rej)

port_rwa_min  = rwa_at_dmin.sum()
port_rwa_mean = rwa_at_dmean.sum()
port_rwa_max  = rwa_at_dmax.sum()

rwa_spread_pct = (port_rwa_max - port_rwa_min) / port_rwa_mean * 100
print(f"Portfolio RWA spread: {rwa_spread_pct:.2f}%")
print(f"  Min vendor:  \u00a3{port_rwa_min:.2f}m  "
      f"({(port_rwa_min/port_rwa_mean - 1)*100:+.1f}% vs mean-damage baseline)")
print(f"  Mean vendor: \u00a3{port_rwa_mean:.2f}m  (baseline)")
print(f"  Max vendor:  \u00a3{port_rwa_max:.2f}m  "
      f"({(port_rwa_max/port_rwa_mean - 1)*100:+.1f}% vs mean-damage baseline)")

cap_min  = compute_capital(port_rwa_min,  cap_ratio)
cap_mean = compute_capital(port_rwa_mean, cap_ratio)
cap_max  = compute_capital(port_rwa_max,  cap_ratio)
cap_uplift_pct = (cap_max - cap_min) / cap_mean * 100
print(f"\nCapital range: {cap_uplift_pct:.2f}% uplift from min-vendor to max-vendor")

# MC portfolio RWA distribution
port_rwa_sim = rwa_sim.sum(axis=1)   # (n_sim,)
rwa_p10, rwa_p50, rwa_p90 = np.percentile(port_rwa_sim, [10, 50, 90])
print(f"\nMC portfolio RWA percentiles (relative to mean-damage baseline):")
print(f"  P10: {(rwa_p10/port_rwa_mean - 1)*100:+.1f}%  "
      f"P50: {(rwa_p50/port_rwa_mean - 1)*100:+.1f}%  "
      f"P90: {(rwa_p90/port_rwa_mean - 1)*100:+.1f}%")

# Lorenz curve / Gini of loan-weighted DRR
def lorenz_gini(values, weights):
    """Lorenz curve coordinates and Gini coefficient."""
    idx   = np.argsort(values)
    v     = values[idx]
    w     = weights[idx]
    cumw  = np.cumsum(w) / w.sum()
    cumvw = np.cumsum(v * w) / (v * w).sum()
    gini  = 1.0 - 2.0 * np.trapz(cumvw, cumw)
    return np.r_[0, cumw], np.r_[0, cumvw], gini

lc_x, lc_y, gini_drr = lorenz_gini(drr, loan_arr)
print(f"\nGini coefficient of loan-weighted DRR: {gini_drr:.4f}")

# Approval rate stacked bar
out_labels    = ["Standard (RW=60%)", "Conditional (RW=75%)", "Reject (RW=105%)"]
bar_colours   = [COLOURS["data_mean"], COLOURS["data_min"], COLOURS["stage2"]]
scenarios     = ["Min vendor\n(d_min)", "Mean vendor\n(d_mean)", "Max vendor\n(d_max)"]
all_counts    = [
    np.bincount(outcome_at_dmin,  minlength=3),
    np.bincount(outcome_at_dmean, minlength=3),
    np.bincount(outcome_at_dmax,  minlength=3),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5.5))

# Panel 1: MC portfolio RWA distribution
ax = axes[0]
ax.hist((port_rwa_sim / port_rwa_mean - 1) * 100, bins=40,
        color=COLOURS["stage1"], edgecolor="white", alpha=0.85)
ax.axvline(0, color="k", lw=1.5, ls="--", label="Mean-damage baseline")
ax.axvline((port_rwa_min  / port_rwa_mean - 1) * 100,
           color=COLOURS["data_min"], lw=1.5, ls=":", label="Min vendor")
ax.axvline((port_rwa_max  / port_rwa_mean - 1) * 100,
           color=COLOURS["data_max"], lw=1.5, ls=":", label="Max vendor")
ax.set_xlabel("Portfolio RWA vs mean-damage baseline (%)")
ax.set_ylabel("Frequency")
ax.set_title("Portfolio RWA distribution (MC)")
ax.legend(fontsize=8)

# Panel 2: Lorenz curve of DRR
ax = axes[1]
ax.plot(lc_x, lc_y, color=COLOURS["triangular"], lw=2.5,
        label=f"DRR Lorenz (Gini={gini_drr:.3f})")
ax.plot([0, 1], [0, 1], color=COLOURS["grey"], lw=1, ls="--", label="Perfect equality")
ax.fill_between(lc_x, lc_y, lc_x, alpha=0.15, color=COLOURS["triangular"])
ax.set_xlabel("Cumulative share of loan book (by EAD)")
ax.set_ylabel("Cumulative share of DRR")
ax.set_title("Lorenz: concentration of reversal risk")
ax.legend(fontsize=9)

# Panel 3: Approval rate stacked bar
ax = axes[2]
bottom = np.zeros(3)
for k, (label, colour) in enumerate(zip(out_labels, bar_colours)):
    vals = np.array([c[k] for c in all_counts], dtype=float)
    ax.bar(scenarios, vals, bottom=bottom, label=label,
           color=colour, edgecolor="white")
    bottom += vals
ax.set_ylabel("Number of loans")
ax.set_title("Outcome by vendor scenario")
ax.legend(fontsize=8, loc="upper left")

plt.suptitle("Tier 3: Portfolio aggregates", y=1.01, fontsize=13, fontweight="bold")
plt.tight_layout()
fig.savefig("../../outputs/figures/decision2/tier3_portfolio_aggregates.pdf", dpi=300)
fig.savefig("../../outputs/figures/decision2/tier3_portfolio_aggregates.png", dpi=300)
plt.show()
print("Saved: tier3_portfolio_aggregates.{pdf,png}")

## 8 — Tier 4: Value of Information

What is the *economic cost* of unresolved vendor uncertainty from a capital planning perspective?

- **Jensen gap per asset**: `E[RWA(d)] − RWA(E[d])` — the difference between the expected RWA under full uncertainty and the RWA at the point estimate. Positive when the RWA function is convex in `d` (i.e. near a threshold).

- **Portfolio EVPVI**: capital-equivalent cost of unresolved vendor uncertainty, computed for (a) at-risk assets (ambiguity zone or DRR > 5%) and (b) the full portfolio.

In [ ]:
# --- Jensen gap per asset ---
# E[RWA(d)] vs RWA(E[d]) — positive indicates convexity of RWA in d near a threshold

e_rwa_under_uncertainty = rwa_sim.mean(axis=0)      # (n_props,)
jensen_gap = e_rwa_under_uncertainty - rwa_at_dmean  # (n_props,)

with np.errstate(invalid="ignore", divide="ignore"):
    jensen_gap_rel = np.where(
        rwa_at_dmean > 0, jensen_gap / rwa_at_dmean * 100, np.nan
    )

print("Jensen gap (E[RWA(d)] \u2212 RWA(E[d])):")
print(f"  Mean per asset (\u00a3):     \u00a3{jensen_gap.mean()*1e3:.2f}k  "
      f"({np.nanmean(jensen_gap_rel):.3f}%)")
print(f"  Max per asset (\u00a3):      \u00a3{jensen_gap.max()*1e3:.2f}k  "
      f"({np.nanmax(jensen_gap_rel):.3f}%)")
print(f"  Portfolio total (\u00a3):    \u00a3{jensen_gap.sum()*1e3:.1f}k  "
      f"({jensen_gap.sum()/rwa_at_dmean.sum()*100:.3f}%)")

# --- Portfolio EVPVI ---
# At-risk definition: in ambiguity zone OR DRR > 5%
at_risk_mask = in_ambig_either | (drr > 0.05)
print(f"\nAt-risk assets (ambiguity zone OR DRR>5%): {at_risk_mask.sum()}")

# Capital cost of uncertainty = MC-mean capital minus point-estimate capital
cap_mc_mean_per_loan = compute_capital(e_rwa_under_uncertainty, cap_ratio)
cap_point_per_loan   = compute_capital(rwa_at_dmean,            cap_ratio)
cap_uncertainty_cost = cap_mc_mean_per_loan - cap_point_per_loan

evpvi_at_risk   = cap_uncertainty_cost[at_risk_mask].sum()
evpvi_portfolio = cap_uncertainty_cost.sum()
print(f"Portfolio EVPVI (capital-equivalent cost of vendor uncertainty):")
print(f"  At-risk assets: \u00a3{evpvi_at_risk*1e3:.1f}k")
print(f"  Full portfolio: \u00a3{evpvi_portfolio*1e3:.1f}k")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
sc = ax.scatter(
    d_mean_arr, jensen_gap_rel,
    c=drr, cmap="hot_r", vmin=0, vmax=drr.max(),
    s=50, alpha=0.75, edgecolors="grey", linewidths=0.3, zorder=3,
)
plt.colorbar(sc, ax=ax, label="DRR")
ax.axhline(0, color="k", lw=1, ls="--")
ax.set_xlabel("Mean damage ratio (d_mean)")
ax.set_ylabel("Jensen gap (% of mean-damage RWA)")
ax.set_title("Tier 4a: Jensen gap vs mean damage ratio")

ax = axes[1]
ax.bar(
    ["At-risk\nassets", "Full\nportfolio"],
    [evpvi_at_risk * 1e3, evpvi_portfolio * 1e3],
    color=[COLOURS["stage2"], COLOURS["stage1"]], edgecolor="white",
)
ax.set_ylabel("Capital-equivalent cost of uncertainty (\u00a3k)")
ax.set_title("Tier 4b: Portfolio EVPVI")

plt.suptitle("Tier 4: Value of information", y=1.01, fontsize=13, fontweight="bold")
plt.tight_layout()
fig.savefig("../../outputs/figures/decision2/tier4_value_of_information.pdf", dpi=300)
fig.savefig("../../outputs/figures/decision2/tier4_value_of_information.png", dpi=300)
plt.show()
print("Saved: tier4_value_of_information.{pdf,png}")

## 9 — θ Sensitivity

Re-run Sections 4–7 core metrics across `theta` sensitivity values from config. θ controls the fraction of physical damage that passes through to collateral value:

- `θ = 1.0` (baseline): full pass-through
- `θ = 0.75`: partial pass-through (e.g. resilience, partial insurance)
- `θ = 0.50`: strong attenuation (e.g. full insurance or resilient construction)

Lower θ compresses the LTV shift under any given damage ratio, moving loans further from the cliff thresholds and reducing both DRR and RWA spread.

In [ ]:
theta_values = cfg_d2["theta"]["sensitivity_values"]
results_theta = []

for th in theta_values:
    # MC transmission for this theta (reuse d_sim)
    v_imp_th  = impaired_collateral_value(v0_arr, d_sim, th)
    ltv_th    = compute_ltv(loan_arr, v_imp_th)
    out_th    = classify_outcome(ltv_th, ltv_bar_1, ltv_bar_2)
    rwa_th    = compute_rwa(loan_arr, out_th, rw_std, rw_cond, rw_rej)

    # Point-estimate outcomes for this theta
    out_mean_th = classify_outcome(
        compute_ltv(loan_arr, impaired_collateral_value(v0_arr, d_mean_arr, th)),
        ltv_bar_1, ltv_bar_2
    )
    out_min_th  = classify_outcome(
        compute_ltv(loan_arr, impaired_collateral_value(v0_arr, d_min_arr, th)),
        ltv_bar_1, ltv_bar_2
    )
    out_max_th  = classify_outcome(
        compute_ltv(loan_arr, impaired_collateral_value(v0_arr, d_max_arr, th)),
        ltv_bar_1, ltv_bar_2
    )
    rwa_mean_th = compute_rwa(loan_arr, out_mean_th, rw_std, rw_cond, rw_rej)
    rwa_min_th  = compute_rwa(loan_arr, out_min_th,  rw_std, rw_cond, rw_rej)
    rwa_max_th  = compute_rwa(loan_arr, out_max_th,  rw_std, rw_cond, rw_rej)

    # DRR
    p_out_th = np.stack([
        (out_th == 0).mean(axis=0),
        (out_th == 1).mean(axis=0),
        (out_th == 2).mean(axis=0),
    ], axis=1)
    drr_th       = 1.0 - p_out_th.max(axis=1)
    drr_mean_th  = (drr_th * weights).sum()

    # RWA spread
    base_rwa_th   = rwa_mean_th.sum()
    rwa_spread_th = (rwa_max_th.sum() - rwa_min_th.sum()) / base_rwa_th * 100

    # Jensen gap (portfolio level, % of mean-damage RWA)
    e_rwa_th    = rwa_th.mean(axis=0)
    jensen_th   = (e_rwa_th - rwa_mean_th).sum() / base_rwa_th * 100

    # Fraction at risk (ambiguity zone)
    d_star_1_th = analytical_damage_thresholds(loan_arr, v0_arr, ltv_bar_1, th)
    d_star_2_th = analytical_damage_thresholds(loan_arr, v0_arr, ltv_bar_2, th)
    in_ambig_th = (
        (~np.isnan(d_star_1_th)) & (d_star_1_th >= d_min_arr) & (d_star_1_th <= d_max_arr) |
        (~np.isnan(d_star_2_th)) & (d_star_2_th >= d_min_arr) & (d_star_2_th <= d_max_arr)
    )
    frac_at_risk = in_ambig_th.mean()

    results_theta.append({
        "theta":            th,
        "drr_portfolio":    drr_mean_th,
        "rwa_spread_pct":   rwa_spread_th,
        "jensen_gap_pct":   jensen_th,
        "frac_at_risk":     frac_at_risk,
    })

df_theta = pd.DataFrame(results_theta)
print("\u03b8 sensitivity results:")
print(df_theta.round(4).to_string(index=False))

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

metric_specs = [
    ("drr_portfolio",  "Portfolio DRR (loan-weighted)",         COLOURS["stage2"]),
    ("rwa_spread_pct", "RWA spread (d_max vs d_min, %)",        COLOURS["beta"]),
    ("jensen_gap_pct", "Jensen gap (% of mean-damage RWA)",     COLOURS["triangular"]),
    ("frac_at_risk",   "Fraction at risk (ambiguity zone)",      COLOURS["uniform"]),
]
for ax, (col, title, colour) in zip(axes.flatten(), metric_specs):
    ax.plot(df_theta["theta"], df_theta[col], "o-",
            color=colour, lw=2.5, ms=8, markeredgecolor="white", markeredgewidth=0.8)
    ax.axvline(cfg_d2["theta"]["baseline"], color="k", ls="--", lw=1, alpha=0.5,
               label=f"Baseline \u03b8={cfg_d2['theta']['baseline']}")
    ax.set_xlabel("Damage pass-through \u03b8")
    ax.set_ylabel(title)
    ax.set_title(title)
    ax.legend(fontsize=9)

plt.suptitle("Section 9: \u03b8 sensitivity — damage pass-through to collateral value",
             y=1.01, fontsize=13, fontweight="bold")
plt.tight_layout()
fig.savefig("../../outputs/figures/decision2/theta_sensitivity.pdf", dpi=300)
fig.savefig("../../outputs/figures/decision2/theta_sensitivity.png", dpi=300)
plt.show()
print("Saved: theta_sensitivity.{pdf,png}")

## 10 — Summary

### Key findings

*[To be populated after running the simulation.]*

- **DRR**: [X]% of the portfolio (by loan count) and [X]% (loan-weighted) experiences meaningful decision reversal risk across the vendor distribution.
- **RWA spread**: Vendor disagreement translates into a [X]% swing in portfolio risk-weighted assets between the most and least conservative vendor draw.
- **Ambiguity zone**: [X] of 100 assets sit in the ambiguity zone where the vendor distribution straddles at least one LTV cliff threshold.
- **Jensen gap**: The expected RWA under full vendor uncertainty exceeds the point-estimate RWA by [X]%, indicating that ignoring uncertainty leads to systematic under-capitalisation for at-risk assets.

### Comparison with Decision 1

Decision 2 has a **shorter transmission chain** than Decision 1: physical damage passes directly through to LTV and risk-weight classification, with no PD or LGD model in between. This makes the cliff mechanics more transparent — any vendor disagreement that causes the impaired LTV to straddle either LTV̄₁ or LTV̄₂ will directly flip the regulatory capital requirement.

Where Decision 1 has a **single SICR cliff** (the EBA doubling rule, τ = 2), Decision 2 has **two LTV cliffs** (LTV̄₁ = 60% and LTV̄₂ = 80%). Loans near either boundary are exposed to reversal risk, and loans between the two boundaries face the possibility of falling through both cliffs (from standard to reject) under sufficiently pessimistic vendor assumptions. The double-cliff structure means that high-LTV₀ loans face a different character of uncertainty than moderate-LTV₀ loans.

The θ sensitivity analysis shows how partial collateral damage pass-through (θ < 1) attenuates cliff exposure: lower θ compresses the stressed LTV distribution, reducing both DRR and RWA spread. This makes θ a natural policy lever — resilience investment or insurance arrangements that reduce physical damage pass-through to market value directly reduce decision-relevant vendor uncertainty.

### Limitations

1. **Static collateral valuation**: The model does not account for pre-sale property devaluation (stigma effects, local market dynamics) or the time-path of collateral value after a flood event.
2. **Independence of vendor draws**: The independent-draw scenario ignores the possibility that vendors use correlated underlying hazard models, which would concentrate portfolio-level uncertainty.
3. **Point-in-time LTV**: The stylised portfolio uses LTV₀ as current LTV. In practice, amortisation would reduce current LTV below origination LTV, pushing more loans below the thresholds.
4. **Conditional framing**: The model conditions on the 1-in-200 year event occurring. A full unconditional framing would weight the damage distribution by the annual exceedance probability (0.5%).
5. **Stylised risk-weight schedule**: The thresholds and risk weights used here (LTV̄₁ = 60%, LTV̄₂ = 80%; RW = 60%/75%/105%) are stylised representations of Basel III SA CRE rules. Actual weights depend on property type, loan purpose, and national supervisor discretion.